# 01. 심화 실습: 선형 회귀로 웨이퍼 수율 예측하기

[00번 기초 실습](00_wafer_inspection_lab.ipynb)을 마친 뒤 진행합니다. 기초에서 확인한 센서 추세와 기준별 적중표를 떠올리되, 여기서 같은 분석을 반복하지 않습니다. **준비 → 모델 학습과 예측 → 평가 → 검증용 적용**에 집중합니다.

## 1. 학습·평가 데이터 준비

Colab의 새 런타임에서는 `연습_3600장.csv`와 `검증_1200장_수율숨김.csv`를 `/content`에 다시 올리고 위에서부터 실행하세요. 로컬에서는 `student-kit/`을 사용합니다. 00번의 변수나 중간 파일은 필요 없습니다.

00번과 동일하게 생산 날짜를 기준으로 앞쪽 2,480장을 학습용, 뒤쪽 1,120장을 평가용으로 나눕니다. 같은 기간을 써야 00번 결과와 비교할 수 있습니다. 수율이 숨겨진 1,200장은 마지막 적용 단계까지 학습·평가에 사용하지 않습니다.

In [ ]:
%pip install -q pandas scikit-learn

In [ ]:
from pathlib import Path
from IPython.display import display
import pandas as pd

practice_name = '연습_3600장.csv'
blind_name = '검증_1200장_수율숨김.csv'
folder = Path('student-kit') if Path('student-kit', practice_name).is_file() else Path.cwd()
practice = pd.read_csv(folder / practice_name, encoding='utf-8-sig')
blind = pd.read_csv(folder / blind_name, encoding='utf-8-sig', dtype={'wafer_id': 'string'})

# 00번과 동일하게 고유 생산 날짜의 앞 70%를 학습에 사용합니다.
sensors = [f'S{number:02d}' for number in range(1, 8)]
practice['production_date'] = pd.to_datetime(practice['production_date'], format='%Y-%m-%d')
dates = practice['production_date'].dropna().drop_duplicates().sort_values().tolist()
if practice['production_date'].isna().any() or len(dates) < 2:
    raise ValueError('빈 생산 날짜가 없어야 하고 서로 다른 날짜가 두 개 이상 필요합니다.')
boundary = dates[max(1, int(len(dates) * 0.7))]
fit = practice.loc[practice['production_date'] < boundary]
compare = practice.loc[practice['production_date'] >= boundary]
print(f'학습 {len(fit):,}장 / 평가 {len(compare):,}장 / 평가 시작일 {boundary:%Y-%m-%d}')

## 2. 선형 회귀 학습과 예측

센서 7개(`S01`~`S07`)가 입력 `X`, 실제 수율(`yield_rate`, %)이 학습용 정답 `y`입니다. 선형 회귀는 센서값에 가중치를 곱해 더한 값으로 수율을 예측합니다. 센서 결측을 채울 중앙값과 회귀 계수는 앞쪽에서만 학습하고, 뒤쪽에서는 센서값만 넣어 예측합니다. `yield_status`는 이 셀에서 사용하지 않습니다.

실행 후 **학습 2,480장 / 수율 예측 1,120장**이 나오는지 확인하세요. 아직 예측이 맞았는지는 확인하지 않았습니다.

In [ ]:
# 센서값은 입력 X, 실제 수율은 학습용 정답 y입니다.
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline

X_train = fit[sensors]
y_train = fit['yield_rate']
X_compare = compare[sensors]

# 결측값을 채울 중앙값과 회귀 계수는 앞쪽 데이터에서만 학습합니다.
model = make_pipeline(SimpleImputer(strategy='median'), LinearRegression())
model.fit(X_train, y_train)
predicted_yield = pd.Series(model.predict(X_compare), index=compare.index)
print(f'학습: {len(X_train):,}장 / 수율 예측: {len(predicted_yield):,}장')

## 3. 모델 평가: 수율 오차와 저수율 적중

모델을 다시 학습하지 않습니다. 2단계의 예측을 두 관점에서 평가합니다. **3.1~3.2는 수율이 얼마나 빗나갔는지**, **3.3~3.4는 검사할 장수가 정해졌을 때 저수율을 얼마나 찾았는지**를 확인합니다.

### 3.1 웨이퍼별 실제값과 예측값 비교하기

**실행 전:** 같은 `wafer_id`의 실제 수율(`yield_rate`)과 예측 수율(`predicted_yield`)을 한 행에 놓습니다. `error`는 예측에서 실제를 뺀 값이고, `absolute_error`는 그 차이의 크기입니다. 먼저 첫 5행에서 한 행을 골라 직접 빼 보세요.

In [ ]:
# 비교 구간에서 웨이퍼별 실제 수율과 예측 수율을 나란히 놓습니다.
evaluation = compare[['wafer_id', 'yield_rate']].copy()
evaluation['predicted_yield'] = predicted_yield

# 방향과 상관없이 얼마나 틀렸는지 보려면 차이의 절댓값을 구합니다.
evaluation['error'] = evaluation['predicted_yield'] - evaluation['yield_rate']
evaluation['absolute_error'] = evaluation['error'].abs()
display(evaluation.head(5).round(2))

**3.1 실행 후:** `error`가 양수면 과대 예측, 음수면 과소 예측입니다. 예를 들어 실제 98%, 예측 95.87%라면 `error`는 약 -2.13%p, `absolute_error`는 2.13%p입니다. 반올림은 미리보기에만 적용합니다.

### 3.2 전체 예측 오차(MAE) 계산하기

**실행 전:** 오차에 부호가 있으면 과대·과소 예측이 서로 상쇄됩니다. 그래서 3.1의 절댓값을 전체 1,120장에 대해 평균 냅니다. 비교 상대는 센서를 보지 않고 모든 웨이퍼에 **앞쪽 학습 구간의 실제 수율 중앙값**을 예측하는 방법입니다. 두 방법을 같은 뒤쪽 데이터에서 비교합니다.

In [ ]:
# 회귀 MAE는 앞 셀에서 구한 웨이퍼별 절대 오차의 평균입니다.
regression_mae = evaluation['absolute_error'].mean()

# 단순 기준은 모든 웨이퍼의 수율을 학습 구간 중앙값으로 예측합니다.
training_median = y_train.median()
baseline_error = (evaluation['yield_rate'] - training_median).abs()
baseline_mae = baseline_error.mean()
print(f'단순 기준이 모든 행에 예측한 수율: {training_median:.2f}%')
print(f'중앙값 기준 MAE: {baseline_mae:.2f}%p')
print(f'선형 회귀 MAE: {regression_mae:.2f}%p')

**3.2 실행 후:** 두 방법의 MAE를 비교하세요. 단위는 %가 아니라 **%p(퍼센트포인트)**입니다. MAE가 2.03%p라면 평가 구간 전체에서 실제 수율과 평균적으로 그만큼 차이 났다는 뜻이지, 모든 웨이퍼에서 같은 오차가 났다는 뜻은 아닙니다.

### 3.3 예측 수율이 낮은 웨이퍼부터 고르기

**실행 전:** 검사할 수 있는 비율을 00번과 동일하게 맞춥니다. 현재는 1,200장 중 100장 비율을 평가용 1,120장에 적용해 93장을 고릅니다. **실제 수율이나 `LOW` 라벨은 보지 않고**, 예측 수율이 낮은 순으로 정렬한 뒤 앞쪽 후보만 가져옵니다.

In [ ]:
# 00번과 같은 검사 비율로, 예측 수율만 사용해 순위를 매깁니다.
if len(blind) < 100:
    raise ValueError('검증용 파일에 웨이퍼가 100장 이상 있어야 합니다.')
budget = max(1, round(len(compare) * 100 / len(blind)))
ranked_yield = predicted_yield.sort_values(kind='stable')
picks = ranked_yield.head(budget).index

selection_preview = compare.loc[picks, ['wafer_id']].copy()
selection_preview['predicted_yield'] = ranked_yield.loc[picks]
print(f'선택: {len(picks)}장')
display(selection_preview.head(5).round(2))

**3.3 실행 후:** 현재 데이터에서는 93개 후보가 나옵니다. 표의 첫 행은 모델이 수율을 가장 낮게 예측한 웨이퍼입니다. 아직 실제 저수율인지 판단한 것은 아닙니다.

### 3.4 실제 라벨을 열어 적중·누락 세기

**실행 전:** 후보를 고른 뒤에만 실제 `LOW` 라벨을 확인합니다. **적중**은 고른 후보 중 실제 `LOW` 수, **누락**은 전체 실제 `LOW` 수에서 적중을 뺀 값입니다. **선택 중 저수율 비율**은 적중을 고른 장수로 나눈 값입니다. 00번의 A·B·무작위 결과도 동일한 93장 기준이므로 적중 수를 직접 비교할 수 있습니다.

In [ ]:
# 후보를 확정한 뒤에만 실제 저수율 라벨을 확인합니다.
low = compare['yield_status'].eq('LOW')
hits = int(low.loc[picks].sum())
misses = int(low.sum()) - hits
print(f'실제 저수율 {low.sum()}장 / 적중 {hits}장 / 누락 {misses}장')
print(f'선택 중 저수율: {hits / len(picks):.1%}')

**해석 예시:** 현재 CSV에서 중앙값 기준 MAE는 2.01%p, 회귀는 2.03%p입니다. 회귀는 93장 중 저수율 10장을 찾았고, 00번 기본 기준 A는 같은 구간에서 24장을 찾았습니다. 이번 데이터에서는 모델을 썼다는 이유만으로 기존 규칙을 바꿀 근거가 없습니다.

## 4. 검증용에 적용하고 결과 해석

모델을 다시 학습하지 않고 수율이 숨겨진 검증용의 센서값을 입력합니다. 예측 수율이 낮은 순서로 우선 검사할 100개 ID를 고릅니다. 화면의 예측은 실제 수율이 아니며, 정답이 공개되기 전에는 적중 수를 알 수 없습니다. 이 단계는 모델 적용 연습이며 00번 제출 CSV를 만들거나 변경하지 않습니다.

**마무리 질문:** 이번 검사 과제에서 회귀와 00번 기준 중 어느 쪽을 선택할지, 같은 검사 예산에서의 적중 수를 근거로 한 문장으로 설명하세요.

In [ ]:
# 학습한 모델로 검증용 수율을 예측하고 낮은 순으로 100장을 고릅니다.
blind_prediction = pd.Series(model.predict(blind[sensors]), index=blind.index)
blind_picks = blind_prediction.sort_values(kind='stable').index[:100]
candidates = blind.loc[blind_picks, ['wafer_id']].copy()
candidates['예측 수율 (%)'] = blind_prediction.loc[blind_picks].round(2)
print(f'검사 후보: {len(candidates)}장 (CSV 저장 안 함)')
display(candidates.head(5).reset_index(drop=True))